In [ ]:
from IPython import display
import ultralytics

In [ ]:
import os

os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"

In [ ]:
import cv2
import matplotlib.pyplot as plt

In [ ]:
display.clear_output()
ultralytics.checks()

In [ ]:
# เรียก best.pt ตรงๆ (ไม่ใช้ yaml.load เพราะ head ของเรามี 3 class ไม่ตรงกับ yolo26n.yaml 80 class)
model = ultralytics.YOLO("runs/detect/rec_stage1/weights/best.pt")

In [ ]:
import glob

imgs = sorted(glob.glob("dataset/rectrain/val/images/*.jpg"))
img = imgs[0]
result = model.predict(img)

In [ ]:
print(len(result[0].boxes))
myimage = cv2.imread(img)
myimage = cv2.cvtColor(myimage, cv2.COLOR_BGR2RGB)
plt.imshow(myimage)

for box in result[0].boxes:
    print(box)
    class_id = result[0].names[box.cls[0].item()]
    print(class_id)
    coor = box.xyxy[0].tolist()
    coors = [round(x) for x in coor]
    print(coors)
    cf = round(box.conf[0].item(), 2)
    print(cf)
    myimage = cv2.rectangle(myimage, (coors[0], coors[1]), (coors[2], coors[3]), (255, 0, 0))
plt.imshow(myimage)

In [ ]:
# Predict ทุกรูปใน val -> เก็บภาพ + CSV ไว้ส่งอาจารย์
import csv
from collections import Counter

COLORS = {"Person": (0, 200, 0), "Car": (255, 140, 0), "Bus": (0, 0, 255)}  # RGB
OUT = "output/final/predict/rec"
os.makedirs(OUT, exist_ok=True)

rows = []
for path in imgs:
    result = model.predict(path, verbose=False)
    im = cv2.imread(path)
    for box in result[0].boxes:
        name = result[0].names[int(box.cls[0].item())]
        x0, y0, x1, y1 = [round(v) for v in box.xyxy[0].tolist()]
        cf = round(box.conf[0].item(), 2)
        bgr = COLORS.get(name, (200, 200, 200))[::-1]
        im = cv2.rectangle(im, (x0, y0), (x1, y1), bgr, 2)
        im = cv2.putText(im, f"{name} {cf}", (x0, max(y0 - 5, 15)), cv2.FONT_HERSHEY_SIMPLEX, 0.6, bgr, 2)
        rows.append([os.path.basename(path), name, cf, x0, y0, x1, y1])
    cv2.imwrite(os.path.join(OUT, os.path.basename(path)), im)

with open("output/final/predict/predict_rec.csv", "w", newline="", encoding="utf-8") as f:
    w = csv.writer(f)
    w.writerow(["image", "class", "conf", "x0", "y0", "x1", "y1"])
    w.writerows(rows)
print(len(imgs), "images,", len(rows), "detections", dict(Counter(r[1] for r in rows)))